In [ ]:
import pandas as pd
import joblib


model = joblib.load("/content/model.pkl")

In [ ]:
import firebase_admin, joblib, pandas as pd, numpy as np
from firebase_admin import credentials, db
from datetime import datetime
import pytz

In [ ]:
FIREBASE_URL = "https://smart-gps-tracker-4bef3-default-rtdb.firebaseio.com/"


In [ ]:
cred = credentials.Certificate("/content/smart-gps-tracker-4bef3-firebase-adminsdk-fbsvc-47a99b3797.json")

In [ ]:
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {"databaseURL": FIREBASE_URL})

In [ ]:
 DEFAULTS = {
     "temperature": 21.0,      # avg ambient temp
#     "battery": 3.6,
#     "distance_percentage": 0.0,
}


def get_season(month):
    if month in (12, 1, 2):
      return "Winter"
    if month in (3, 4, 5):
      return "Spring"
    if month in (6, 7, 8):
      return "Summer"
    return "Fall"


In [ ]:
def firebase_to_feature_row(bin_data: dict) -> pd.DataFrame:

    fill = bin_data.get("fillLevel", 0.0)
    dist = bin_data.get("distanceFromSensor", 0.0)
    lat  = bin_data.get("latitude", 0.0)
    lng  = bin_data.get("longitude", 0.0)
    gass = bin_data.get("gasLevel", 0.0)


    tstr = bin_data.get("readableTime")
    try:
        dt = datetime.strptime(tstr, "%Y-%m-%d %H:%M:%S")
    except Exception:
        dt = datetime.utcnow()

    month       = dt.month
    hour        = dt.hour
    weekday     = dt.weekday()
    is_weekend  = 1 if weekday >= 5 else 0
    season      = get_season(month)

    FEATURES = [
    "temperature",
    "distance",
    "fillLevel",
    "Lat",
    "Long",
    "gas_sensor",
    "month",
    "hour",
    "day_of_week",
    "is_weekend",
    "season",
    ]

    row = {
      "temperature":        bin_data.get("temperature", DEFAULTS["temperature"]),
      "distance":           dist,
      "fillLevel":          fill,
      "Lat":                lat,
      "Long":               lng,
      "gas_sensor":         gass,
      "month":              month,
      "hour":               hour,
      "day_of_week":        weekday,
      "is_weekend":         is_weekend,
      "season":             season,
    }

    # 4) Return as 1×12 DataFrame
    return pd.DataFrame([row], columns=FEATURES)

In [ ]:
root = db.reference("smartWasteBins")
all_bins = root.get()

In [ ]:
bins_to_collect = []
now = datetime.utcnow().isoformat()

In [ ]:
for bin_id, payload in all_bins.items():
    current = payload.get("currentData", {})

    features_df = firebase_to_feature_row(current)
    urgency = model.predict(features_df)[0]

    db.reference(f"smartWasteBins/{bin_id}/prediction").set(
        {
        "fillLevel": current.get("fillLevel"),
        "urgency": urgency,
        "predictedAt": now,
        "latitude": current.get("latitude"),
        "longitude": current.get("longitude")})


    if urgency in ["within_1hr", "1-3hrs", "3-6hrs"]:
        bins_to_collect.append({
            "bin_id": bin_id,
            "urgency": urgency,
            "latitude": current.get("latitude"),
            "longitude": current.get("longitude"),
        })
print(f"Processing Done. Predictions uploaded to Firebase")

Processing Done. Predictions uploaded to Firebase


Note: Route optimization has been moved client‑side in JavaScript for zero‑cost, real‑time calculation. This notebook now only handles data ingestion, ML prediction, and exporting to Firebase.